# Project 57 — Local Expense Processing Agent
## Receipt Parsing → Categorization → Policy Check → Report

**Stack:** LangChain · Ollama · Pydantic · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic pandas

## Step 1 — Define Expense Schemas

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
import json, pandas as pd

llm = ChatOllama(model="qwen3:8b", temperature=0.0)

class ExpenseItem(BaseModel):
    vendor: str
    amount: float
    currency: str = Field(default="USD")
    category: str = Field(description="meals, travel, lodging, supplies, software, other")
    date: str
    description: str
    reimbursable: bool
    receipt_confidence: float = Field(ge=0, le=1, description="Parsing confidence")

class PolicyViolation(BaseModel):
    rule: str
    violation: str
    severity: str = Field(description="warning, violation, block")

class ExpenseReport(BaseModel):
    employee: str
    period: str
    items: list[ExpenseItem]
    violations: list[PolicyViolation]
    total_amount: float
    total_reimbursable: float

extractor = llm.with_structured_output(ExpenseReport)
print("Expense processor ready!")

## Step 2 — Process Receipts

In [ ]:
receipts = [
    "Uber ride from SFO to downtown hotel — $48.50 — Jan 15, 2025 — business travel for client meeting",
    "Team dinner at Nobu — $385.00 — Jan 15, 2025 — 4 attendees, client entertainment",
    "Marriott Hotel — 2 nights — $520.00 — Jan 15-17, 2025 — conference accommodation",
    "AWS monthly — $1,250.00 — Jan 2025 — dev infrastructure",
    "Office Depot — $67.30 — Jan 18, 2025 — printer toner, paper, pens",
    "Starbucks — $6.75 — Jan 15, 2025 — morning coffee",
    "Delta Airlines — $890.00 — Jan 14, 2025 — roundtrip SFO-NYC economy",
    "Apple store — $1,299.00 — Jan 20, 2025 — MacBook charger and accessories",
]

receipt_block = "\n".join([f"Receipt {i+1}: {r}" for i, r in enumerate(receipts)])

report = extractor.invoke(
    f"Employee: John Smith\nPeriod: January 2025\n\n"
    f"Company policy: meals under $75/person, flights must be economy, "
    f"hotel max $300/night, purchases over $500 need pre-approval.\n\n"
    f"Parse these receipts into a full expense report:\n{receipt_block}"
)

print(f"EXPENSE REPORT — {report.employee}")
print(f"Period: {report.period}")
print(f"{'='*60}")
for item in report.items:
    flag = "✓" if item.reimbursable else "✗"
    print(f"  {flag}  ${item.amount:>9,.2f}  {item.category:<10}  {item.vendor} — {item.description[:40]}")

print(f"\nTotal:         ${report.total_amount:,.2f}")
print(f"Reimbursable:  ${report.total_reimbursable:,.2f}")

## Step 3 — Policy Violation Report

In [ ]:
if report.violations:
    print("POLICY VIOLATIONS")
    print("=" * 50)
    for v in report.violations:
        icon = {"warning":"🟡", "violation":"🔴", "block":"⛔"}[v.severity]
        print(f"  {icon} [{v.severity.upper()}] {v.rule}")
        print(f"     {v.violation}")
else:
    print("No policy violations detected")

# Category summary
df = pd.DataFrame([item.model_dump() for item in report.items])
print("\nSPENDING BY CATEGORY")
print(df.groupby("category")["amount"].agg(["sum","count"]).sort_values("sum", ascending=False).to_string())

## Step 4 — Approval Recommendation

In [ ]:
approval_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a finance manager reviewing expense reports. "
     "Provide an approval decision: approve, approve_with_notes, or reject. "
     "Explain your reasoning."),
    ("human", "Total: ${total:,.2f}\nReimbursable: ${reimb:,.2f}\n"
     "Violations: {violations}\nCategories: {categories}")
])
approval_chain = approval_prompt | llm | StrOutputParser()

decision = approval_chain.invoke({
    "total": report.total_amount,
    "reimb": report.total_reimbursable,
    "violations": json.dumps([v.model_dump() for v in report.violations]),
    "categories": df.groupby("category")["amount"].sum().to_dict() if len(df) > 0 else {},
})
print("APPROVAL DECISION")
print("=" * 50)
print(decision)

## What You Learned
- **Multi-stage receipt processing**: parse → categorize → policy-check → approve
- **Policy violation detection** with structured output
- **Financial reporting** with pandas aggregation
- **Approval workflow** with LLM reasoning